# Transport 1D Postprocessing

This notebook demonstrates the portable `solution.transport_1d` API on the geometric-pinch Bohm-gyroBohm example.

It shows how to:
- inspect the saved reduced and coefficient groups
- plot raw saved 1D transport coefficients in adimensional units
- plot the effective dimensional coefficients actually used by the solver
- project those effective coefficients back to 2D mesh views


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from hdg_postprocess.api import configure_solution_setup, load_solution


In [ ]:
solution_path = "demos/data/solutions/limiter_case/steady_pinch_geometric_gyrobohm/"
solution_base = "Sol2D_WEST_60527_P4_DPe0.100E+02_DPai0.314E+06_DPae0.105E+08"
reference_element = "demos/data/reference_elements/reference_triangle_P4.mat"

solution = load_solution(solution_path, solution_base, n_partitions=1)
configure_solution_setup(solution, reference_element=reference_element, neutral_diffusion=True)


In [ ]:
{
    "profiles": sorted(solution.transport_1d.profiles.keys()),
    "coefficients": sorted(solution.transport_1d.coefficients.keys()),
    "params": sorted(solution.transport_1d.params.keys()),
}


## Raw Saved 1D Coefficients

These are the saved transport-model outputs on the stored `rho_grid`, before runtime blending and windowing is applied.


In [ ]:
raw = solution.transport_1d.coefficient_profiles(effective=False, dimensional=False)

fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
axes = axes.ravel()
for ax, key in zip(axes, ["chi_i_fs", "chi_e_fs", "d_fs", "nu_mom_fs", "vpinch_fs"]):
    ax.plot(raw["rho_grid"], raw[key], lw=2)
    ax.set_xlabel(r"$\rho_{pol,norm}$")
    ax.set_title(f"{key} (raw, adim)")
    ax.grid(alpha=0.3)
axes[-1].axis("off")
plt.show()


## Effective 1D Coefficients Used By The Solver

These include the Fortran-side diffusion blending, floors, and pinch windows reconstructed from the saved parameters.


In [ ]:
effective = solution.transport_1d.coefficient_profiles(effective=True, dimensional=True)

labels = {
    "chi_i_fs": r"$\chi_i$ [m$^2$/s]",
    "chi_e_fs": r"$\chi_e$ [m$^2$/s]",
    "d_fs": r"$D$ [m$^2$/s]",
    "nu_mom_fs": r"$\nu_{mom}$ [m$^2$/s]",
    "vpinch_fs": r"$v_{pinch}$ [m/s]",
}

fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
axes = axes.ravel()
for ax, key in zip(axes, ["chi_i_fs", "chi_e_fs", "d_fs", "nu_mom_fs", "vpinch_fs"]):
    ax.plot(effective["rho_grid"], effective[key], lw=2)
    ax.set_xlabel(r"$\rho_{pol,norm}$")
    ax.set_ylabel(labels[key])
    ax.set_title(f"{key} (effective, dimensional)")
    ax.grid(alpha=0.3)
axes[-1].axis("off")
plt.show()


## Project Effective Coefficients To 2D

The projected fields match the chosen solution view. `simple` gives nodal plotting data compatible with `mesh.plot.full(..., connectivity=mesh.geometry.connectivity_big)`.


In [ ]:
projected = solution.transport_1d.coefficient_profiles(view="simple", effective=True, dimensional=True)
connectivity_big = solution.mesh.geometry.connectivity_big

fig, axes = plt.subplots(2, 3, figsize=(14, 9), constrained_layout=True)
axes = axes.ravel()
for ax, key in zip(axes, ["chi_i_fs", "chi_e_fs", "d_fs", "nu_mom_fs", "vpinch_fs"]):
    solution.mesh.plot.full(
        projected[key],
        ax=ax,
        connectivity=connectivity_big,
        n_levels=80,
        label=labels[key],
    )
    ax.set_title(key)
axes[-1].axis("off")
plt.show()


In [ ]:
{
    "simple_shape": solution.transport_1d.project("d_fs", view="simple").shape,
    "full_shape": solution.transport_1d.project("d_fs", view="full").shape,
    "gauss_shape": solution.transport_1d.project("d_fs", view="gauss").shape,
}
